# Building Damage Detection

Train, validate, and run YOLOv8n on the Roboflow xView2 v4 export. See README for environment setup.

In [ ]:
import os
from pathlib import Path

from roboflow import Roboflow

api_key = os.environ.get("ROBOFLOW_API_KEY")
if not api_key:
    raise RuntimeError("Set ROBOFLOW_API_KEY before starting Jupyter.")
project = Roboflow(api_key=api_key).workspace("ozu").project("xview2")
dataset = project.version(4).download("yolov8")
data_yaml = Path(dataset.location).resolve() / "data.yaml"
if not data_yaml.is_file():
    raise FileNotFoundError(data_yaml)


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data=str(data_yaml),
    epochs=20,
    imgsz=768,
    lr0=0.02,
    batch=16,
    patience=10,
    project=str(Path("runs") / "detect"),
    name="xview2-damage-detector",
)
checkpoint = Path(model.trainer.best)
trained_model = YOLO(str(checkpoint))


In [ ]:
metrics = trained_model.val(data=str(data_yaml), split="val", plots=True)
print(metrics.results_dict)


## Optional prediction

Set the path below to an image you can use.

In [ ]:
image_path = Path("inputs/example.jpg")
if image_path.is_file():
    predictions = trained_model.predict(source=str(image_path), save=True)
    print(f"Predictions saved to: {predictions[0].save_dir}")
else:
    print(f"Add an image at {image_path} or change image_path to try prediction.")
